In [ ]:
# Market Risk — baseline: fetch, ingest, VaR/ES, DCC-GARCH, backtests
import json
import os
import time
from datetime import datetime, timedelta
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests as re
import scipy as sp
from dotenv import load_dotenv
from massive import RESTClient

import risk_utils as ru

%matplotlib inline
plt.rcParams["figure.dpi"] = 300
load_dotenv()

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
Path("images").mkdir(exist_ok=True)



In [ ]:
# --- Configuration ---
LOOKBACK_DAYS = 720
OUTPUT_STOCKS = "data/portfolio_stocks_2yrs_hist_dret.csv"
OUTPUT_BONDS = "data/portfolio_bonds_2yrs_hist_dret.csv"
OUTPUT_LOG_RETS = "data/portfolio_log_returns_2y.csv"
OUTPUT_PF_LOG_RETS = "data/portfolio_pf_log_returns_2y.csv"
OUTPUT_MANIFEST = "data/data_manifest.json"

BOND_TICKERS = {"1YR": "yield_1_year", "5YR": "yield_5_year", "10YR": "yield_10_year"}
FRED_SERIES = {"1YR": "DGS1", "5YR": "DGS5", "10YR": "DGS10"}

ALPHA = ru.ALPHA
EST_WINDOW = 252
ROLL_STEP = 1

today = datetime.today()
start_date = (today - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")
end_date = today.strftime("%Y-%m-%d")
print(f"Window: {start_date} -> {end_date}")



In [ ]:
# S&P 500 ticker list (Wikipedia)
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
header = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36",
    "X-Requested-With": "XMLHttpRequest",
}
r = re.get(url, headers=header, timeout=30)
table = pd.read_html(StringIO(r.text))
tickers = list(table[0]["Symbol"])
STOCK_TICKERS = tickers[:15] + ["GLD", "SLV"]
print(f"Equity/ETF universe ({len(STOCK_TICKERS)}):", STOCK_TICKERS)



In [ ]:
# --- Fetch equities/ETFs (Polygon) ---
client = RESTClient(api_key=os.getenv("MASSIVE_API_KEY"))
rows = []

for i, ticker in enumerate(STOCK_TICKERS):
    print(f"Downloading {i + 1}/{len(STOCK_TICKERS)}: {ticker}")
    for a in client.list_aggs(
        ticker, 1, "day", start_date, end_date, adjusted="true", sort="asc"
    ):
        date = datetime.fromtimestamp(int(a.timestamp / 1000)).strftime("%Y-%m-%d")
        rows.append([date, ticker, a.open, a.high, a.low, a.close, a.volume])
    if i % 4 == 0 and i != 0:
        time.sleep(60)

stocks_df = pd.DataFrame(rows, columns=["date", "ticker", "open", "high", "low", "close", "volume"])
stocks_df = stocks_df.drop_duplicates(subset=["date", "ticker"])
stocks_df.to_csv(OUTPUT_STOCKS, index=False)
print(f"Wrote {OUTPUT_STOCKS}: {len(stocks_df)} rows, tickers={stocks_df['ticker'].nunique()}")



In [ ]:
# --- Fetch Treasury yields (Polygon, paginated) ---
rows = []
seen_dates = set()
batch = list(
    client.list_treasury_yields(
        date_gte=start_date, date_lte=end_date, limit=50000, sort="date.asc"
    )
)
for date_row in batch:
    d = date_row.date
    if d in seen_dates:
        continue
    seen_dates.add(d)
    for k, attr in BOND_TICKERS.items():
        rows.append([d, k, getattr(date_row, attr)])

bonds_polygon = pd.DataFrame(rows, columns=["date", "ticker", "rate"])
bonds_polygon.to_csv(OUTPUT_BONDS, index=False)
print(f"Wrote {OUTPUT_BONDS}: {len(bonds_polygon)} rows from Polygon")



In [ ]:
# --- FRED backup / cross-check (optional) ---
def fetch_fred_yields(start: str, end: str) -> pd.DataFrame:
    api_key = os.getenv("FRED_API_KEY")
    rows = []
    if api_key:
        import pandas_datareader as pdr

        for label, series in FRED_SERIES.items():
            s = pdr.get_data_fred(series, start=start, end=end, api_key=api_key)
            s = s.reset_index().rename(columns={"DATE": "date", series: "rate"})
            s["ticker"] = label
            rows.append(s[["date", "ticker", "rate"]])
    else:
        import pandas_datareader as pdr

        for label, series in FRED_SERIES.items():
            s = pdr.get_data_fred(series, start=start, end=end)
            s = s.reset_index()
            date_col = "DATE" if "DATE" in s.columns else s.columns[0]
            s = s.rename(columns={date_col: "date", series: "rate"})
            s["date"] = pd.to_datetime(s["date"]).dt.strftime("%Y-%m-%d")
            s["ticker"] = label
            rows.append(s[["date", "ticker", "rate"]])
    out = pd.concat(rows, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"]).dt.strftime("%Y-%m-%d")
    return out.dropna(subset=["rate"])


try:
    bonds_fred = fetch_fred_yields(start_date, end_date)
    merged = bonds_polygon.merge(
        bonds_fred,
        on=["date", "ticker"],
        how="outer",
        suffixes=("_polygon", "_fred"),
    )
    both = merged.dropna(subset=["rate_polygon", "rate_fred"])
    if len(both):
        diff = (both["rate_polygon"] - both["rate_fred"]).abs()
        print(f"FRED vs Polygon: {len(both)} overlapping rows, max |diff|={diff.max():.4f}")
    missing_poly = bonds_fred[
        ~bonds_fred.set_index(["date", "ticker"]).index.isin(
            bonds_polygon.set_index(["date", "ticker"]).index
        )
    ]
    if len(missing_poly):
        print(f"Filling {len(missing_poly)} bond rows from FRED only")
        bonds_df = pd.concat([bonds_polygon, missing_poly], ignore_index=True)
        bonds_df.to_csv(OUTPUT_BONDS, index=False)
except Exception as e:
    print(f"FRED backup skipped: {e}")

bonds_df = pd.read_csv(OUTPUT_BONDS)
bonds_df.head()



In [ ]:
# TradingView manual ingest helper (optional; not mixed into 2y panel by default)

def ingest_tradingview(path: str, ticker: str) -> pd.DataFrame:
    return ru.load_tradingview_csv(path, ticker)

# Example: tv = ingest_tradingview('data/GLD_5yrs_historical_prices.csv', 'GLD')



In [ ]:
# --- Ingest: aligned log-return panel ---
stocks_df = pd.read_csv(OUTPUT_STOCKS)
bonds_df = pd.read_csv(OUTPUT_BONDS)
stocks_df["date"] = pd.to_datetime(stocks_df["date"])
bonds_df["date"] = pd.to_datetime(bonds_df["date"])

stocks_dates = set(stocks_df["date"].unique())
bonds_dates = set(bonds_df["date"].unique())
dates = sorted(stocks_dates & bonds_dates)
print(f"Aligned trading days: {len(dates)} ({dates[0].date()} .. {dates[-1].date()})")

asset_names = list(STOCK_TICKERS) + list(BOND_TICKERS.keys())
cols = []
stocks_df = stocks_df[stocks_df["date"].isin(dates)]
for ticker in STOCK_TICKERS:
    col = stocks_df[stocks_df["ticker"] == ticker].sort_values("date")["close"].to_numpy()
    cols.append(col)
stock_prices = np.stack(cols, axis=1)
stock_rets = stock_prices[1:, :] / stock_prices[:-1, :]

cols = []
bonds_df = bonds_df[bonds_df["date"].isin(dates)]
for ticker in BOND_TICKERS.keys():
    col = bonds_df[bonds_df["ticker"] == ticker].sort_values("date")["rate"].to_numpy()
    cols.append(col)
bond_rates = np.stack(cols, axis=1)
bond_rets = ((bond_rates[1:, :] - bond_rates[:-1, :]) / 100) + 1

rets = np.hstack([stock_rets, bond_rets])
logrets = np.log(rets)
return_dates = pd.DatetimeIndex(dates)[1:]

log_df = pd.DataFrame(logrets, columns=asset_names)
log_df.insert(0, "date", return_dates.strftime("%Y-%m-%d"))
log_df.to_csv(OUTPUT_LOG_RETS, index=False)

weights = np.ones(logrets.shape[1]) / logrets.shape[1]
pf_logrets = (logrets @ weights.reshape(-1, 1)).flatten()
pf_df = pd.DataFrame({"date": return_dates.strftime("%Y-%m-%d"), "log_return": pf_logrets})
pf_df.to_csv(OUTPUT_PF_LOG_RETS, index=False)

# Quality gates
nan_pct = np.isnan(logrets).mean(axis=0)
for name, pct in zip(asset_names, nan_pct):
    if pct > 0.05:
        print(f"WARNING: {name} missing {pct*100:.1f}%")
assert "GLD" in asset_names and "SLV" in asset_names
assert not np.isnan(pf_logrets).any()

manifest = {
    "generated_at": datetime.now().isoformat(),
    "start_date": str(return_dates[0].date()),
    "end_date": str(return_dates[-1].date()),
    "n_return_days": int(len(pf_logrets)),
    "n_assets": len(asset_names),
    "assets": asset_names,
    "sources": {"equities": "polygon_massive_adjusted", "bonds": "polygon_treasury_fred_backup"},
    "files": {
        "stocks": OUTPUT_STOCKS,
        "bonds": OUTPUT_BONDS,
        "log_returns": OUTPUT_LOG_RETS,
        "portfolio_log_returns": OUTPUT_PF_LOG_RETS,
    },
}
with open(OUTPUT_MANIFEST, "w") as f:
    json.dump(manifest, f, indent=2)
print("Manifest written:", OUTPUT_MANIFEST)
log_df.head()



In [ ]:
# Portfolio return EDA
ys = pf_logrets
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(ys, bins=50)
ax1.set_title("Equal-weight portfolio log returns")
ax2.plot(return_dates, ys)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()



In [ ]:
# Distribution fits (portfolio)
mu, std = sp.stats.norm.fit(pf_logrets)
df_t, loc, scale = sp.stats.t.fit(pf_logrets)
kde = sp.stats.gaussian_kde(pf_logrets.T)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 8))
x = np.linspace(pf_logrets.min(), pf_logrets.max(), 200)
for ax, pdf, title in [
    (ax1, sp.stats.norm.pdf(x, mu, std), "Normal"),
    (ax2, sp.stats.t.pdf(x, df_t, loc, scale), "Student-t"),
    (ax3, kde(x), "KDE"),
]:
    ax.hist(pf_logrets, bins=50, density=True, alpha=0.6)
    ax.plot(x, pdf, "k")
    ax.set_title(title)
ax4.set_visible(False)
plt.tight_layout()
plt.savefig("images/portfolio-return-distributions.png", bbox_inches="tight")
plt.show()



In [ ]:
# --- In-sample VaR / ES (99%) ---
methods = {}

var_log, es_log = ru.historical_var_es(pf_logrets, ALPHA)
methods["historical"] = {"var_log": var_log, "es_log": es_log}

var_log, es_log = ru.parametric_var_es_normal(pf_logrets, ALPHA)
methods["normal"] = {"var_log": var_log, "es_log": es_log}

var_log, es_log = ru.parametric_var_es_t(pf_logrets, ALPHA)
methods["t"] = {"var_log": var_log, "es_log": es_log}

summary = []
for name, m in methods.items():
    summary.append(
        {
            "method": name,
            "VaR_99_loss_pct": ru.log_var_to_loss(m["var_log"]) * 100,
            "ES_99_loss_pct": ru.log_var_to_loss(m["es_log"]) * 100,
        }
    )
pd.DataFrame(summary)



In [ ]:
# --- DCC-GARCH (fallback: GARCH-t on portfolio if full DCC fails) ---
from arch import arch_model

dcc_methods = {}
try:
    # Reduced universe for multivariate stability
    dcc_assets = ["GLD", "SLV", "1YR", "5YR", "10YR", "A", "MSFT"] if "MSFT" in asset_names else ["GLD", "SLV", "1YR", "5YR", "10YR", "A", "ADBE"]
    dcc_assets = [a for a in dcc_assets if a in asset_names]
    sub = log_df[dcc_assets].to_numpy()
    resid = np.zeros_like(sub)
    vols = []
    for j in range(sub.shape[1]):
        am = arch_model(sub[:, j] * 100, mean="Constant", vol="GARCH", p=1, q=1, dist="t")
        res = am.fit(disp="off")
        resid[:, j] = res.resid / res.conditional_volatility
        vols.append(res.conditional_volatility[-1] / 100)
    corr = np.corrcoef(resid.T)
    w = np.ones(len(dcc_assets)) / len(dcc_assets)
    port_vol = float(np.sqrt(w @ (np.diag(vols) @ corr @ np.diag(vols)) @ w))
    port_mean = float(sub.mean(axis=0) @ w)
    var_log = port_mean + sp.stats.t.ppf(ALPHA, df=8) * port_vol
    tail = sp.stats.t.ppf(ALPHA, df=8)
    es_factor = -sp.stats.t.pdf(tail, df=8) / ALPHA * (8 + tail**2) / 7
    es_log = port_mean + es_factor * port_vol
    dcc_methods["dcc_garch"] = {"var_log": var_log, "es_log": es_log, "assets": dcc_assets}
    print("DCC-GARCH (reduced universe):", dcc_assets)
except Exception as e:
    print(f"DCC reduced fit failed ({e}); using univariate GARCH-t on portfolio")

am = arch_model(pf_logrets * 100, mean="Constant", vol="GARCH", p=1, q=1, dist="t")
garch_res = am.fit(disp="off")
fcast = garch_res.forecast(horizon=1, reindex=False)
mean_h = float(garch_res.params.get("mu", garch_res.params.get("Const", 0))) / 100
vol_h = float(np.sqrt(fcast.variance.values[-1, 0])) / 100
nu = float(garch_res.params.get("nu", 8))
var_log = mean_h + sp.stats.t.ppf(ALPHA, df=nu) * vol_h
tail = sp.stats.t.ppf(ALPHA, df=nu)
es_log = mean_h + (-sp.stats.t.pdf(tail, df=nu) / ALPHA * (nu + tail**2) / (nu - 1)) * vol_h
dcc_methods["garch_t_portfolio"] = {"var_log": var_log, "es_log": es_log}

for name, m in dcc_methods.items():
    summary.append(
        {
            "method": name,
            "VaR_99_loss_pct": ru.log_var_to_loss(m["var_log"]) * 100,
            "ES_99_loss_pct": ru.log_var_to_loss(m["es_log"]) * 100,
        }
    )
pd.DataFrame(summary)



In [ ]:
# --- Rolling OOS VaR / ES backtests ---
def rolling_var_es(
    log_returns: np.ndarray,
    method: str,
    est_window: int = EST_WINDOW,
) -> tuple[np.ndarray, np.ndarray]:
    n = len(log_returns)
    var_path = np.full(n, np.nan)
    es_path = np.full(n, np.nan)
    for t in range(est_window, n):
        sample = log_returns[t - est_window : t]
        if method == "historical":
            v, e = ru.historical_var_es(sample, ALPHA)
        elif method == "t":
            v, e = ru.parametric_var_es_t(sample, ALPHA)
        elif method == "garch":
            try:
                am = arch_model(sample * 100, mean="Constant", vol="GARCH", p=1, q=1, dist="t")
                res = am.fit(disp="off")
                fc = res.forecast(horizon=1, reindex=False)
                mean_h = float(res.params.get("mu", 0)) / 100
                vol_h = float(np.sqrt(fc.variance.values[-1, 0])) / 100
                nu = float(res.params.get("nu", 8))
                v = mean_h + sp.stats.t.ppf(ALPHA, df=nu) * vol_h
                tail = sp.stats.t.ppf(ALPHA, df=nu)
                e = mean_h + (-sp.stats.t.pdf(tail, df=nu) / ALPHA * (nu + tail**2) / (nu - 1)) * vol_h
            except Exception:
                v, e = ru.parametric_var_es_t(sample, ALPHA)
        else:
            v, e = ru.parametric_var_es_normal(sample, ALPHA)
        var_path[t] = v
        es_path[t] = e
    return var_path, es_path


oos_methods = ["historical", "t", "garch"]
backtest_rows = []
bt_results = {}

for method in oos_methods:
    var_path, es_path = rolling_var_es(pf_logrets, method)
    valid = ~np.isnan(var_path)
    realized = pf_logrets[valid]
    v = var_path[valid]
    e = es_path[valid]
    breach = realized <= v
    bt_results[method] = {
        "var_log": v,
        "es_log": e,
        "breach": breach,
        "dates": return_dates[valid],
    }
    kup = ru.kupiec_test(int(breach.sum()), len(breach))
    cc = ru.christoffersen_conditional_coverage(breach)
    bl = ru.basel_traffic_light(breach)
    mf = ru.mcneil_frey_es_test(realized, float(np.median(v)), float(np.median(e)))
    backtest_rows.append(
        {
            "method": method,
            "kupiec_lr": kup["lr"],
            "kupiec_p": kup["p_value"],
            "kupiec_reject": kup["reject_5pct"],
            "cc_lr": cc["lr"],
            "cc_p": cc["p_value"],
            "cc_reject": cc["reject_5pct"],
            "basel_green_pct": bl["green_pct"],
            "basel_yellow_pct": bl["yellow_pct"],
            "basel_red_pct": bl["red_pct"],
            "es_violation_rate": mf["es_violation_rate"],
            "n_oos": len(breach),
            "breach_rate": float(breach.mean()),
        }
    )

backtest_df = pd.DataFrame(backtest_rows)
backtest_df.to_csv("data/portfolio_backtest_summary.csv", index=False)
backtest_df



In [ ]:
# Plot rolling VaR (t method)
method = "t"
v = bt_results[method]["var_log"]
d = bt_results[method]["dates"]
loss_var = np.array([ru.log_var_to_loss(x) * 100 for x in v])
r_loss = -(np.exp(pf_logrets[EST_WINDOW:]) - 1) * 100

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(d, r_loss, alpha=0.5, label="Realized loss %")
ax.plot(d, loss_var, color="r", label="Rolling 99% VaR %")
ax.set_title("Rolling OOS VaR — Student-t")
ax.legend()
plt.tight_layout()
plt.savefig("images/portfolio-rolling-var-t.png", bbox_inches="tight")
plt.show()

